# Customer Segmentation & Marketing Analytics

**Objective:** Segment customers using Python and clustering techniques so that marketing teams can design more targeted customer strategies.

### Business questions
1. What are the main customer spending and engagement patterns?
2. Which customer attributes are useful for segmentation?
3. How many customer segments provide a practical balance between separation and interpretability?
4. What business action should be considered for each segment?

**Workflow:** Data audit → cleaning → feature engineering → EDA → scaling → K-Means → cluster validation → PCA visualization → business interpretation.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)


## 1. Load and audit the data

The dataset contains 2,240 customer records and 29 variables covering demographics, purchase behavior, channel activity and campaign responses.

In [ ]:
data = pd.read_csv("marketing_campaign.csv", sep=";")

print("Rows:", data.shape[0])
print("Columns:", data.shape[1])
display(data.head())
display(data.isna().sum().sort_values(ascending=False).head(10))


## 2. Data cleaning

Income has missing observations. Very old birth-year values are treated as data-quality outliers for the age-based analysis. Missing-income customers are excluded because income is one of the clustering variables.

In [ ]:
df = data.dropna(subset=["Income"]).copy()

df = df[
    (df["Year_Birth"] >= 1920) &
    (df["Year_Birth"] <= 2000)
].copy()

print("Rows after cleaning:", len(df))


## 3. Feature engineering

New variables translate raw fields into business-friendly measures:

- **Age**
- **TotalSpend**
- **TotalPurchases**
- **CampaignAccepted**
- **HouseholdDependents**
- **WebEngagement**


In [ ]:
df["Age"] = 2015 - df["Year_Birth"]

spend_cols = [
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds"
]
df["TotalSpend"] = df[spend_cols].sum(axis=1)

purchase_cols = [
    "NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"
]
df["TotalPurchases"] = df[purchase_cols].sum(axis=1)

campaign_cols = [
    "AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
    "AcceptedCmp4", "AcceptedCmp5"
]
df["CampaignAccepted"] = df[campaign_cols].sum(axis=1)

df["HouseholdDependents"] = df["Kidhome"] + df["Teenhome"]
df["WebEngagement"] = df["NumWebPurchases"] + df["NumWebVisitsMonth"]

display(
    df[[
        "ID", "Age", "Income", "TotalSpend",
        "TotalPurchases", "CampaignAccepted",
        "WebEngagement", "HouseholdDependents"
    ]].head()
)


## 4. Exploratory analysis

We first inspect how spending, income and purchase activity vary before applying clustering.

In [ ]:
summary = df[
    ["Age", "Income", "Recency", "TotalSpend",
     "TotalPurchases", "CampaignAccepted", "WebEngagement"]
].describe().T

display(summary)


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist(df["TotalSpend"], bins=35)
plt.xlabel("Total spend")
plt.ylabel("Customers")
plt.title("Distribution of Customer Spending")
plt.tight_layout()
plt.show()


## 5. Prepare features for clustering

K-Means is distance-based, so the variables are standardized. Income and TotalSpend are capped at their 1st and 99th percentiles to reduce the influence of extreme values without deleting otherwise valid customers.

In [ ]:
cluster_features = [
    "Income", "Recency", "TotalSpend", "TotalPurchases",
    "CampaignAccepted", "WebEngagement", "HouseholdDependents"
]

X = df[cluster_features].copy()

for col in ["Income", "TotalSpend"]:
    low, high = X[col].quantile([0.01, 0.99])
    X[col] = X[col].clip(low, high)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Clustering features:", cluster_features)


## 6. Select the number of clusters

The elbow curve shows the reduction in inertia as k increases. Silhouette score provides an additional measure of separation. Four clusters are selected as a practical, interpretable solution for the business analysis.

In [ ]:
k_values = range(2, 9)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

results = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette_score": silhouettes
})

display(results)


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Analysis")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(list(k_values), silhouettes, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Analysis")
plt.tight_layout()
plt.show()


## 7. Build the customer segments

The final model uses **k = 4**. The goal is not to claim that four is mathematically the only answer; it is a practical business segmentation that produces distinct and interpretable profiles.

In [ ]:
k = 4

model = KMeans(n_clusters=k, random_state=42, n_init=20)
df["Segment"] = model.fit_predict(X_scaled)

segment_profile = df.groupby("Segment")[cluster_features].mean().round(1)
segment_counts = df["Segment"].value_counts().sort_index()

display(segment_profile)
display(segment_counts.rename("Customers"))


## 8. Reduce dimensions for visualization

PCA compresses the standardized clustering features into two dimensions so that the resulting segments can be viewed visually. PCA is used here for visualization, not as a replacement for the business features used by K-Means.

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_values = pca.fit_transform(X_scaled)

df["PCA1"] = pca_values[:, 0]
df["PCA2"] = pca_values[:, 1]

plt.figure(figsize=(8, 5))
for segment in sorted(df["Segment"].unique()):
    part = df[df["Segment"] == segment]
    plt.scatter(
        part["PCA1"], part["PCA2"],
        s=20, alpha=0.55, label=f"Segment {segment}"
    )

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Customer Segments in PCA Space")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Business interpretation

The segments should be interpreted from their average income, spending, purchase activity, campaign response and digital engagement rather than from the cluster number itself.

Example interpretations from this analysis:

- **Premium High-Value:** highest spending and income; suitable for loyalty, premium offers and retention.
- **Low-Engagement:** lower spending and purchase activity; suitable for activation and introductory offers.
- **Established Value:** high spending with lower digital engagement; suitable for cross-channel and loyalty strategies.
- **Digital-Engaged:** strong purchase activity and the highest web engagement; suitable for personalized digital campaigns.

These labels are business interpretations, not model-generated truths.

In [ ]:
profile = df.groupby("Segment")[cluster_features].mean().round(1)
profile["Customers"] = df["Segment"].value_counts().sort_index()
profile["SharePct"] = (profile["Customers"] / len(df) * 100).round(1)

display(profile)


## 10. Export the analysis

The final customer-level segment file can be used for further reporting or a Power BI dashboard.

In [ ]:
output_columns = [
    "ID", "Age", "Education", "Marital_Status", "Income",
    "Recency", "TotalSpend", "TotalPurchases",
    "CampaignAccepted", "WebEngagement",
    "HouseholdDependents", "Segment", "PCA1", "PCA2"
]

df[output_columns].to_csv("customer_segments.csv", index=False)

print("Saved: customer_segments.csv")


## Key takeaway

Customer segmentation converts product and customer-level behavior into actionable groups. The strongest business value comes from connecting each segment to a different marketing, retention or engagement strategy rather than treating every customer the same.